In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from tqdm import tqdm

# -----------------------------
# 1. CONFIG
# -----------------------------
CSV_PATH = "juliet_cwe_dataset_sample.csv"  # or "juliet_cwe_dataset.csv"
MODEL_NAME = "Salesforce/codet5-small"
MAX_LENGTH = 256
BATCH_SIZE = 4
EPOCHS = 2
LR = 5e-5

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"🚀 Using device: {device}")


/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Using device: mps


In [2]:
# ============================================
# 🧠 PHASE 3: MODEL TRAINING & EVALUATION
# Multi-class CWE Classification (118 classes)
# Model: CodeT5-small
# ============================================

import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from tqdm import tqdm

# -----------------------------
# 1. CONFIG
# -----------------------------
CSV_PATH = "/Users/rohan/Documents/Final year/cwe_project/scripts/juliet_cwe_sample.csv"
MODEL_NAME = "Salesforce/codet5-small"
MAX_LENGTH = 256
BATCH_SIZE = 4
EPOCHS = 2
LR = 5e-5

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"🚀 Using device: {device}")

# -----------------------------
# 2. LOAD DATA
# -----------------------------
df = pd.read_csv(CSV_PATH)

# Keep only needed columns
df = df[["code_clean", "cwe_index"]].dropna()
df["cwe_index"] = df["cwe_index"].astype(int)

print("📦 Dataset loaded")
print(f"Total samples: {len(df)}")
print(df["cwe_index"].value_counts().head())

# Train/validation split (stratified by CWE)
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["cwe_index"]
)

print(f"\n🧪 Train samples: {len(train_df)}")
print(f"🧪 Val samples:   {len(val_df)}")

# -----------------------------
# 3. TOKENIZER
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("\n✅ Tokenizer loaded")

# -----------------------------
# 4. CUSTOM DATASET
# -----------------------------
class JulietDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        code = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            code,
            truncation=True,
            padding="max_length",
            max_length=self.max_len
        )

        item = {k: torch.tensor(v) for k, v in encoding.items()}
        item["labels"] = torch.tensor(label)
        return item

train_dataset = JulietDataset(
    train_df["code_clean"],
    train_df["cwe_index"],
    tokenizer,
    max_len=MAX_LENGTH
)

val_dataset = JulietDataset(
    val_df["code_clean"],
    val_df["cwe_index"],
    tokenizer,
    max_len=MAX_LENGTH
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("\n✅ Datasets & dataloaders ready")

# -----------------------------
# 5. MODEL & OPTIMIZER
# -----------------------------
num_labels = df["cwe_index"].nunique()
print(f"\n🔢 Number of CWE classes: {num_labels}")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=LR)
print("✅ Model & optimizer ready")

# -----------------------------
# 6. TRAINING LOOP
# -----------------------------
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def eval_model(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / len(dataloader)
    return avg_loss, all_labels, all_preds


for epoch in range(EPOCHS):
    print(f"\n================= EPOCH {epoch+1}/{EPOCHS} =================")

    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    print(f"📉 Train loss: {train_loss:.4f}")

    val_loss, y_true, y_pred = eval_model(model, val_loader, device)
    print(f"📉 Val loss:   {val_loss:.4f}")

    print("\n📊 Classification Report (Validation):")
    print(classification_report(y_true, y_pred, digits=4))

    print("📊 Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

print("\n✅ Training complete!")

# -----------------------------
# 7. SAVE MODEL
# -----------------------------
SAVE_DIR = "codet5_juliet_multiclass"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"\n💾 Model & tokenizer saved to: {SAVE_DIR}")

🚀 Using device: mps
📦 Dataset loaded
Total samples: 2065
cwe_index
2      163
1      149
110    148
11     122
7       91
Name: count, dtype: int64

🧪 Train samples: 1652
🧪 Val samples:   413


/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



✅ Tokenizer loaded

✅ Datasets & dataloaders ready

🔢 Number of CWE classes: 118


W1219 13:18:50.431000 37062 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Some weights of T5ForSequenceClassification were not initialized from the model checkpoint at Salesforce/codet5-small and are newly initialized: ['classification_head.dense.bias', 'classification_head.dense.weight', 'classification_head.out_proj.bias', 'classification_head.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


✅ Model & optimizer ready

================= EPOCH 1/2 =================


...

📉 Val loss:   1.5671

📊 Classification Report (Validation):
              precision    recall  f1-score   support

           0     1.0000    0.6667    0.8000         3
           1     0.9091    1.0000    0.9524        30
           2     0.9706    1.0000    0.9851        33
           3     0.0000    0.0000    0.0000         1
           4     1.0000    0.9167    0.9565        12
           5     0.0769    0.1250    0.0952         8
           6     0.4000    1.0000    0.5714        12
           7     0.6667    1.0000    0.8000        18
           8     0.0000    0.0000    0.0000         1
           9     0.0000    0.0000    0.0000         1
          10     0.0000    0.0000    0.0000         1
          11     0.7576    1.0000    0.8621        25
          12     0.7826    1.0000    0.8780        18
          13     0.5833    1.0000    0.7368         7
          14     0.8333    0.7143    0.7692         7
          15     0.0000    0.0000    0.0000         1
          16     0.62

Training: 100%|██████████| 413/413 [03:59<00:00,  1.72it/s]


📉 Train loss: 1.2964


Evaluating: 100%|██████████| 104/104 [00:15<00:00,  6.64it/s]
/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. 

📉 Val loss:   0.6108

📊 Classification Report (Validation):
              precision    recall  f1-score   support

           0     1.0000    0.6667    0.8000         3
           1     1.0000    1.0000    1.0000        30
           2     0.9706    1.0000    0.9851        33
           3     0.5000    1.0000    0.6667         1
           4     1.0000    1.0000    1.0000        12
           5     1.0000    1.0000    1.0000         8
           6     1.0000    1.0000    1.0000        12
           7     0.9000    1.0000    0.9474        18
           8     0.0000    0.0000    0.0000         1
           9     1.0000    1.0000    1.0000         1
          10     0.0000    0.0000    0.0000         1
          11     1.0000    1.0000    1.0000        25
          12     1.0000    1.0000    1.0000        18
          13     0.8750    1.0000    0.9333         7
          14     0.7778    1.0000    0.8750         7
          15     0.0000    0.0000    0.0000         1
          16     0.83

In [3]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_labels), np.array(all_preds)


y_true, y_pred = evaluate_model(model, val_loader, device)

print("\n📊 Classification Report:")
print(classification_report(
    y_true, y_pred,
    digits=4,
))


📊 Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.6667    0.8000         3
           1     1.0000    1.0000    1.0000        30
           2     0.9706    1.0000    0.9851        33
           3     0.5000    1.0000    0.6667         1
           4     1.0000    1.0000    1.0000        12
           5     1.0000    1.0000    1.0000         8
           6     1.0000    1.0000    1.0000        12
           7     0.9000    1.0000    0.9474        18
           8     0.0000    0.0000    0.0000         1
           9     1.0000    1.0000    1.0000         1
          10     0.0000    0.0000    0.0000         1
          11     1.0000    1.0000    1.0000        25
          12     1.0000    1.0000    1.0000        18
          13     0.8750    1.0000    0.9333         7
          14     0.7778    1.0000    0.8750         7
          15     0.0000    0.0000    0.0000         1
          16     0.8333    1.0000    0.9091         5
 

/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war